In [284]:
import matplotlib.pyplot as plt
import math
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import os
import pickle
import time
import logging
import getpass
import random
import yaml

from scipy.interpolate import griddata

In [285]:
# %%
def load_datapath(filepath):
    with open(filepath, "r") as file:
        config = yaml.safe_load(file)
    return config["paths"]["progress_dir"]

# %%
def get_magnitudes(momentum_list):
    magnitude_list = []
    for p in momentum_list:
        magnitude_list.append(np.linalg.norm(p.values, axis=1)[0])
    return magnitude_list

def top_results(X, y, n=1):
    #watch out that X, y have same indexing
    X_der = X.copy().reset_index(drop=True)
    y_der = y.copy().reset_index(drop=True)
    ycolumn = y_der.columns[0]
    top_indices = y_der.sort_values(by=ycolumn, ascending=True).head(n).index.tolist()
    top_y = y_der.loc[top_indices].reset_index(drop=True)
    top_X = X_der.loc[top_indices].reset_index(drop=True)
    return top_X, top_y

In [286]:
def load_data(dragon_datapath):
    logging.info("\n")
    logging.info("-"*20+"Loading and processing new data"+"-"*20)

    # 1. Load data and save in pd.DataFrame
    data_loaded = False
    while not data_loaded:
        try:
            with open(dragon_datapath, "rb") as file:
                foundfiles,globpardict,pflxs,pfs,x2s,x2dict,vx2s,vx2dict,rx2s,rx2dict,px2s,px2dict,hx2s,hx2dict,fx2s,fx2dict,qx2s,qx2dict,vfx2s,vfx2dict,likes,likedict,mlikes,mlikedict,tlikes,tlikedict,qlikes,qlikedict,tss,tsdict,faultfiles,faultpardict,corrfactlist = pickle.load(file)
                globpardict["Y"]=likes
                data = pd.DataFrame.from_dict(globpardict)
                faultdata = pd.DataFrame.from_dict(faultpardict)
            data_loaded = True
        except Exception as e:
            logging.warning(f"Error occured: {e}")
            logging.info("Two runs might be trying to access data simultaneously. Waiting before trying again.")
            time.sleep(random.randint(0, 300)) 

    logging.info(f"Total simulations loaded: {len(data)+len(faultdata)}")
    logging.info(f"Number failed simulations: {len(faultfiles)}")


    # 2. Process data by dropping nans and infs  
    init_len = len(data)
    data.replace([np.inf, -np.inf], np.nan, inplace=True)
    faultdata = pd.concat([faultdata, data[data.isna().any(axis=1)].copy()], ignore_index=True, axis=0).drop(columns=["Y"]).reset_index(drop=True)
    ###### Improvement possibility: handle inf/nan/fails differently by assigning high neg. log-likelihood ######
    data = data.dropna().reset_index(drop=True)
    post_len = len(data)
    
    logging.info(f"Number inf/nan simulations: {(init_len-post_len)}")
    logging.info(f"Final selected simulations: {post_len}")
    logging.info(f"Best neg. log-likelihood: { np.min(data['Y']) }")


    # 3. Fitting scalers to standardize data
    y_data = data[["Y"]].copy()
    X_data = data.copy().drop(columns=y_data.columns)

    Xscaler= StandardScaler()
    yscaler = StandardScaler()
    X_scaled = pd.DataFrame(Xscaler.fit_transform(X_data.copy()), columns=X_data.columns)
    y_scaled = pd.DataFrame(yscaler.fit_transform(y_data.copy()), columns=y_data.columns)
    
    return X_data, y_data, X_scaled, y_scaled, Xscaler, yscaler, faultdata

In [287]:
def closest_points(X, y, point, n):
    X_der = X.copy().reset_index(drop=True)
    y_der = y.copy().reset_index(drop=True)
    point_der = point.copy().iloc[0]
    distances = np.linalg.norm(X_der-point_der, axis=1)

    closest_indices = np.argsort(distances)[:n]
    closest_X = X_der.loc[closest_indices].reset_index(drop=True)
    closest_y = y_der.loc[closest_indices].reset_index(drop=True)
    return closest_X, closest_y

In [288]:
def create_heatmap(X, y, best_X, save_folder):
    d=len(X.columns)
    num_plots = math.ceil(d/2)
    fig, axes = plt.subplots(math.ceil(num_plots/3), 3, figsize=(16, 4*math.ceil(num_plots/3)))
    
    
    for i in range(math.ceil(num_plots/3)):
        for j in range(3):
            ax = axes[i, j]
            cur_d = (i*3+j)*2
            if cur_d < d-1:
                dim1 = X.columns[cur_d]
                dim2 = X.columns[cur_d+1]
            else:
                dim1 = X.columns[cur_d]
                dim2 = X.columns[0]
             
            # sc = ax.scatter(X[dim1], X[dim2], marker="o", color = "black", label="Points")
            sc = ax.scatter(X[dim1], X[dim2], c=y["Y"], marker="o", cmap="viridis", edgecolor="k", label="Points")
            ax.scatter(best_X[dim1], best_X[dim2], marker="*", s=150, color = "r", label="Best Point")
            ax.set_xlabel(dim1)
            ax.set_ylabel(dim2)
            
             
            xi = np.linspace(np.min(X[dim1]), np.max(X[dim1]), 100)
            yi = np.linspace(np.min(X[dim2]), np.max(X[dim2]), 100)
            xi, yi = np.meshgrid(xi, yi)

            zi = griddata((X[dim1].values.flatten(), X[dim2].values.flatten()), y.values.flatten(), (xi, yi), method="cubic")
            heatmap = ax.imshow(zi, extent=[np.min(X[dim1]), np.max(X[dim1]), np.min(X[dim2]), np.max(X[dim2])], origin="lower", cmap="viridis", alpha=0.9)
            cbar =fig.colorbar(sc, ax=ax, label = "NLL")
            
            ax.set_xlim(np.min(X[dim1]), np.max(X[dim1]))
            ax.set_ylim(np.min(X[dim2]), np.max(X[dim2]))
            ax.set_aspect("auto")
            
       
    plt.legend()
    plt.suptitle("Neg. Log-Likelihood Heat Map")      
    plt.tight_layout() 
    plt.savefig(os.path.join(save_folder, "heatmap.png"))      
    plt.close()
    pass

In [289]:
# %%
def main():
    config_filepath = f"/home/{getpass.getuser()}/CALETana/MLProject/BayesianOptimization/config.yaml"
    dragondata_path = f"/home/{getpass.getuser()}/CALETana/astro/CALplots/fitspectra-HeP_fitsave-5.0.dat"
    
    save_folder = f"/home/{getpass.getuser()}/CALETana/MLProject/ComparisonPlots/plots"
    X_data, y_data, X_scaled, y_scaled, Xscaler, yscaler, X_fault = load_data(dragondata_path)
    
    best_X, best_y = top_results(X_data, y_data, 1)
    nX, ny = closest_points(X_data, y_data, best_X, 50)
    
    create_heatmap(nX, ny, best_X, save_folder)
    
main()